# 3일차 팀 프로젝트: CVE·CWE 보안 취약점 Text2SQL 시스템

Day 2의 보안 취약점 RAG 주제를 관계형 데이터 분석으로 확장합니다.

구현 흐름
1. CVE/CWE CSV 자동 탐색 및 품질 검증
2. Supabase의 기존 대문자 테이블 CVE, CWE 자동 탐색
3. CVE의 복수 CWE 값을 동적으로 분해하여 JOIN
4. 읽기 전용 SQL 실행 및 안전성 검증
5. 자연어 질문 → PostgreSQL → 실행 결과 → 자연어 답변

이 노트북은 Supabase 데이터를 수정하지 않으며 SELECT와 WITH 쿼리만 허용합니다.

## 0. 환경 및 CSV 경로 자동 탐색

In [ ]:
import os
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display


def find_project_root(start: Path) -> Path:
    """노트북 실행 위치와 무관하게 rag-system 루트를 찾는다."""
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "examples").is_dir():
            return candidate
    raise FileNotFoundError("pyproject.toml이 있는 rag-system 경로를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root(Path.cwd())
ENV_PATH = PROJECT_ROOT / ".env"
load_dotenv(ENV_PATH, override=True, encoding="utf-8-sig")


def first_existing_path(env_name: str, candidates: list[Path]) -> Path | None:
    """환경 변수 경로를 우선하고, 없으면 후보 중 첫 파일을 선택한다."""
    configured = os.getenv(env_name)
    search_paths = ([Path(configured).expanduser()] if configured else []) + candidates
    return next((path.resolve() for path in search_paths if path.is_file()), None)


desktop = Path.home() / "Desktop"
CVE_CSV_PATH = first_existing_path(
    "CVE_CSV_PATH",
    [
        PROJECT_ROOT / "datasets" / "cve.csv",
        desktop / "cve.csv",
    ],
)
CWE_CSV_PATH = first_existing_path(
    "CWE_CSV_PATH",
    [
        PROJECT_ROOT / "datasets" / "cwe.csv",
        desktop / "2000.csv" / "cwe.csv",
        desktop / "cwe.csv",
    ],
)

if CVE_CSV_PATH is None or CWE_CSV_PATH is None:
    raise FileNotFoundError(
        "CVE/CWE CSV를 찾지 못했습니다. CVE_CSV_PATH와 CWE_CSV_PATH 환경 변수를 설정하세요."
    )
if not os.getenv("SUPABASE_DB_URL"):
    raise EnvironmentError(f"SUPABASE_DB_URL이 없습니다. {ENV_PATH}를 확인하세요.")

print(f"프로젝트 루트: {PROJECT_ROOT}")
print(f"CVE CSV: {CVE_CSV_PATH}")
print(f"CWE CSV: {CWE_CSV_PATH}")
print("✓ Supabase DB URL 확인 완료 (접속 정보는 출력하지 않음)")

## 1. CSV 로딩과 데이터 품질 검증

In [ ]:
cve_df = pd.read_csv(CVE_CSV_PATH, dtype=str, keep_default_na=False, low_memory=False)
# CWE 원본은 일부 행 끝에 헤더보다 하나 많은 빈 필드가 있어,
# 열 수를 명시하지 않으면 pandas가 CWE-ID를 행 인덱스로 잘못 해석할 수 있다.
cwe_column_count = len(pd.read_csv(CWE_CSV_PATH, nrows=0).columns)
cwe_df = pd.read_csv(
    CWE_CSV_PATH,
    dtype=str,
    keep_default_na=False,
    low_memory=False,
    index_col=False,
    usecols=range(cwe_column_count),
)
cve_df.columns = cve_df.columns.str.strip()
cwe_df.columns = cwe_df.columns.str.strip()

required_cve_columns = {
    "cveID", "vendorProject", "product", "vulnerabilityName", "dateAdded",
    "shortDescription", "requiredAction", "dueDate",
    "knownRansomwareCampaignUse", "notes", "cwes",
}
required_cwe_columns = {
    "CWE-ID", "Name", "Weakness Abstraction", "Status", "Description",
}
missing_cve_columns = sorted(required_cve_columns - set(cve_df.columns))
missing_cwe_columns = sorted(required_cwe_columns - set(cwe_df.columns))
if missing_cve_columns or missing_cwe_columns:
    raise ValueError(
        f"필수 컬럼 누락: CVE={missing_cve_columns}, CWE={missing_cwe_columns}"
    )

cve_id_valid = cve_df["cveID"].str.fullmatch(r"CVE-\d{4}-\d{4,}", na=False)
cwe_id_valid = cwe_df["CWE-ID"].str.fullmatch(r"\d+", na=False)
quality = {
    "CVE 행": len(cve_df),
    "CWE 행": len(cwe_df),
    "CVE ID 중복": int(cve_df["cveID"].duplicated().sum()),
    "CWE ID 중복": int(cwe_df["CWE-ID"].duplicated().sum()),
    "잘못된 CVE ID": int((~cve_id_valid).sum()),
    "잘못된 CWE ID": int((~cwe_id_valid).sum()),
    "CWE가 없는 CVE": int((~cve_df["cwes"].str.contains(r"CWE-\d+", regex=True)).sum()),
}
display(pd.DataFrame([quality]))

assert quality["CVE ID 중복"] == 0
assert quality["CWE ID 중복"] == 0
assert quality["잘못된 CVE ID"] == 0
assert quality["잘못된 CWE ID"] == 0
print("✓ CSV 기본키와 필수 컬럼 검증 통과")

print("\nCVE 샘플")
display(cve_df[["cveID", "vendorProject", "product", "dateAdded", "cwes"]].head(3))
print("CWE 샘플")
display(cwe_df[["CWE-ID", "Name", "Weakness Abstraction", "Status"]].head(3))

In [ ]:
# CVE 한 행의 복수 CWE 값을 관계 형태로 펼쳐 데이터 결합 가능성을 점검한다.
cve_cwe_map = (
    cve_df[["cveID", "cwes"]]
    .assign(cwe_id=lambda frame: frame["cwes"].str.findall(r"CWE-\d+"))
    .explode("cwe_id")
    .dropna(subset=["cwe_id"])
    .rename(columns={"cveID": "cve_id"})[["cve_id", "cwe_id"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
catalog_ids = {f"CWE-{value}" for value in cwe_df["CWE-ID"]}
unmapped_ids = sorted(set(cve_cwe_map["cwe_id"]) - catalog_ids)
multi_cwe_count = int(
    cve_df["cwes"].str.findall(r"CWE-\d+").map(len).gt(1).sum()
)

print(f"CVE–CWE 관계 행: {len(cve_cwe_map):,}개")
print(f"두 개 이상의 CWE를 가진 CVE: {multi_cwe_count:,}개")
print(f"현재 CWE 카탈로그에 없는 레거시 CWE ID: {len(unmapped_ids)}개")
if unmapped_ids:
    print("  " + ", ".join(unmapped_ids))
print("✓ 다중 CWE는 SQL에서 쉼표 기준으로 동적 분해하여 JOIN합니다.")

## 2. Supabase 연결 및 실제 스키마 자동 탐색

사용자가 만든 public.CVE와 public.CWE 테이블을 대소문자 구분 없이 찾은 뒤 실제 이름을 사용합니다.

In [ ]:
from langchain_community.utilities import SQLDatabase
from sqlalchemy import create_engine, inspect, text

engine = create_engine(
    os.environ["SUPABASE_DB_URL"],
    pool_pre_ping=True,
    connect_args={"connect_timeout": 10, "options": "-c statement_timeout=15000"},
)
inspector = inspect(engine)
public_tables = inspector.get_table_names(schema="public")


def resolve_table(expected: str) -> str:
    matches = [name for name in public_tables if name.casefold() == expected.casefold()]
    if len(matches) != 1:
        raise LookupError(f"public 스키마에서 {expected} 테이블을 하나 찾지 못했습니다: {matches}")
    return matches[0]


CVE_TABLE = resolve_table("CVE")
CWE_TABLE = resolve_table("CWE")
table_columns = {
    table: [column["name"] for column in inspector.get_columns(table, schema="public")]
    for table in (CVE_TABLE, CWE_TABLE)
}

if not required_cve_columns.issubset(table_columns[CVE_TABLE]):
    raise ValueError("Supabase CVE 테이블의 컬럼이 CSV와 일치하지 않습니다.")
if not required_cwe_columns.issubset(table_columns[CWE_TABLE]):
    raise ValueError("Supabase CWE 테이블의 컬럼이 CSV와 일치하지 않습니다.")

db = SQLDatabase(
    engine,
    schema="public",
    include_tables=[CVE_TABLE, CWE_TABLE],
    sample_rows_in_table_info=0,
)

print(f"✓ 연결된 테이블: public.{CVE_TABLE}, public.{CWE_TABLE}")
for table, columns in table_columns.items():
    print(f"  - {table}: {len(columns)}개 컬럼")

In [ ]:
def quote_identifier(value: str) -> str:
    return '"' + value.replace('"', '""') + '"'


def qualified_table(table: str) -> str:
    return f'{quote_identifier("public")}.{quote_identifier(table)}'


with engine.connect() as connection:
    transaction = connection.begin()
    connection.execute(text("SET TRANSACTION READ ONLY"))
    db_counts = {}
    for table in (CVE_TABLE, CWE_TABLE):
        db_counts[table] = connection.execute(
            text(f"SELECT COUNT(*) FROM {qualified_table(table)}")
        ).scalar_one()
    transaction.rollback()

comparison = pd.DataFrame(
    [
        {"데이터": "CVE", "CSV 행": len(cve_df), "Supabase 행": db_counts[CVE_TABLE]},
        {"데이터": "CWE", "CSV 행": len(cwe_df), "Supabase 행": db_counts[CWE_TABLE]},
    ]
)
comparison["일치"] = comparison["CSV 행"] == comparison["Supabase 행"]
display(comparison)
assert comparison["일치"].all(), "CSV와 Supabase 행 수가 다릅니다. 업로드 상태를 확인하세요."
print("✓ CSV와 Supabase 행 수 일치")

## 3. 읽기 전용 SQL 실행기

생성 SQL은 SELECT 또는 WITH로 시작해야 하며, 쓰기·DDL 명령과 다중 문장을 차단합니다. 실행 트랜잭션도 PostgreSQL 읽기 전용으로 설정합니다.

In [ ]:
MAX_RESULT_ROWS = 100
FORBIDDEN_SQL = re.compile(
    r"\b(INSERT|UPDATE|DELETE|DROP|ALTER|CREATE|TRUNCATE|MERGE|GRANT|REVOKE|"
    r"COPY|CALL|EXECUTE|VACUUM|ANALYZE|COMMENT|REFRESH|ATTACH|DETACH)\b",
    flags=re.IGNORECASE,
)


def clean_sql(raw_sql: str) -> str:
    sql = str(raw_sql).strip()
    fence = chr(96) * 3
    if sql.startswith(fence):
        lines = sql.splitlines()
        sql = "\n".join(lines[1:-1]).strip()
    sql = re.sub(r"/\*.*?\*/", " ", sql, flags=re.DOTALL)
    sql = re.sub(r"--[^\n]*", " ", sql)
    sql = sql.strip()
    if sql.endswith(";"):
        sql = sql[:-1].strip()
    return sql


def validate_read_only_sql(raw_sql: str) -> str:
    sql = clean_sql(raw_sql)
    if ";" in sql:
        raise ValueError("다중 SQL 문장은 실행할 수 없습니다.")
    if not re.match(r"^(SELECT|WITH)\b", sql, flags=re.IGNORECASE):
        raise ValueError("SELECT 또는 WITH 쿼리만 실행할 수 있습니다.")
    if FORBIDDEN_SQL.search(sql):
        raise ValueError("허용되지 않은 SQL 명령이 포함되어 있습니다.")
    if re.search(r"\b(information_schema|pg_catalog|pg_\w+)\b", sql, re.IGNORECASE):
        raise ValueError("시스템 카탈로그 조회는 허용하지 않습니다.")
    return sql


def run_read_only_sql(raw_sql: str, max_rows: int = MAX_RESULT_ROWS) -> pd.DataFrame:
    sql = validate_read_only_sql(raw_sql)
    limited_sql = f'SELECT * FROM ({sql}) AS "_limited_result" LIMIT {int(max_rows)}'
    with engine.connect() as connection:
        transaction = connection.begin()
        try:
            connection.execute(text("SET TRANSACTION READ ONLY"))
            connection.execute(text("SET LOCAL statement_timeout = '15s'"))
            result = pd.read_sql_query(text(limited_sql), connection)
        finally:
            transaction.rollback()
    return result


# 안전성 자체 테스트
for unsafe in ("DELETE FROM public.\"CVE\"", "SELECT 1; DROP TABLE x"):
    try:
        validate_read_only_sql(unsafe)
        raise AssertionError("위험 SQL 차단 실패")
    except ValueError:
        pass
print("✓ 읽기 전용 SQL 검증 테스트 통과")

## 4. 직접 작성한 SQL 테스트

기본 조회, 집계, CVE–CWE JOIN, 매핑 품질 검사를 실제 Supabase에서 실행합니다.

In [ ]:
SQL_TESTS = {
    "최근 등록 취약점": r'''
        SELECT "cveID", "vendorProject", "product", "dateAdded", "dueDate"
        FROM public."CVE"
        ORDER BY "dateAdded"::date DESC, "cveID"
        LIMIT 5
    ''',
    "제조사별 취약점 수": r'''
        SELECT "vendorProject", COUNT(*) AS vulnerability_count
        FROM public."CVE"
        GROUP BY "vendorProject"
        ORDER BY vulnerability_count DESC, "vendorProject"
        LIMIT 10
    ''',
    "가장 많이 연결된 CWE": r'''
        SELECT
            'CWE-' || w."CWE-ID"::text AS cwe_id,
            w."Name" AS weakness_name,
            COUNT(DISTINCT v."cveID") AS vulnerability_count
        FROM public."CVE" AS v
        JOIN LATERAL regexp_split_to_table(
            COALESCE(v."cwes", ''), '[[:space:]]*,[[:space:]]*'
        ) AS mapping(cwe_id) ON TRUE
        JOIN public."CWE" AS w
          ON mapping.cwe_id = 'CWE-' || w."CWE-ID"::text
        GROUP BY w."CWE-ID", w."Name"
        ORDER BY vulnerability_count DESC, w."CWE-ID"
        LIMIT 10
    ''',
    "CWE 카탈로그 미매핑 ID": r'''
        SELECT mapping.cwe_id, COUNT(DISTINCT v."cveID") AS vulnerability_count
        FROM public."CVE" AS v
        JOIN LATERAL regexp_split_to_table(
            COALESCE(v."cwes", ''), '[[:space:]]*,[[:space:]]*'
        ) AS mapping(cwe_id) ON TRUE
        LEFT JOIN public."CWE" AS w
          ON mapping.cwe_id = 'CWE-' || w."CWE-ID"::text
        WHERE mapping.cwe_id <> '' AND w."CWE-ID" IS NULL
        GROUP BY mapping.cwe_id
        ORDER BY mapping.cwe_id
    ''',
}

sql_test_results = {}
for name, sql in SQL_TESTS.items():
    result = run_read_only_sql(sql)
    sql_test_results[name] = result
    print(f"\n[{name}] {len(result)}행")
    display(result)

assert not sql_test_results["최근 등록 취약점"].empty
assert not sql_test_results["제조사별 취약점 수"].empty
assert not sql_test_results["가장 많이 연결된 CWE"].empty
print("✓ Supabase 직접 SQL 4종 실행 완료")

## 5. 동적 Text2SQL 구현

LLM에는 실제 Supabase 스키마와 정확한 CVE–CWE JOIN 규칙을 제공합니다. OpenAI 호출이 불가능한 환경에서는 대표 질문을 처리하는 검증용 규칙 기반 폴백으로 계속 실행됩니다.

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

SCHEMA_DESCRIPTION = "\n".join(
    f'public.{quote_identifier(table)}: ' + ", ".join(quote_identifier(column) for column in columns)
    for table, columns in table_columns.items()
)
JOIN_GUIDE = r'''
CVE.cwes는 "CWE-94, CWE-352"처럼 복수 ID가 들어갈 수 있다.
CVE와 CWE를 연결할 때 반드시 아래 구조를 사용한다.

FROM public."CVE" AS v
JOIN LATERAL regexp_split_to_table(
    COALESCE(v."cwes", ''), '[[:space:]]*,[[:space:]]*'
) AS mapping(cwe_id) ON TRUE
JOIN public."CWE" AS w
  ON mapping.cwe_id = 'CWE-' || w."CWE-ID"::text
'''

OFFLINE_TEST = os.getenv("DAY3_OFFLINE_TEST", "0").lower() in {"1", "true", "yes"}
CHAT_MODEL = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.4-mini")
llm = None
if os.getenv("OPENAI_API_KEY") and not OFFLINE_TEST:
    try:
        llm = init_chat_model(CHAT_MODEL)
        print(f"✓ Text2SQL 모델 준비: {CHAT_MODEL}")
    except Exception as error:
        print(f"⚠ 모델 초기화 실패, 규칙 기반 폴백 사용: {type(error).__name__}")
else:
    print("ℹ 오프라인 테스트 모드: 규칙 기반 Text2SQL 사용")


def fallback_sql(question: str) -> str:
    """네트워크 없이도 프로젝트 대표 질문을 검증하는 PostgreSQL 생성기."""
    q = question.strip()
    cwe_match = re.search(r"CWE[- ]?(\d+)", q, re.IGNORECASE)
    if cwe_match:
        cwe_number = int(cwe_match.group(1))
        return f'''
            SELECT v."cveID", v."vendorProject", v."product",
                   w."Name" AS weakness_name, v."dateAdded"
            FROM public."CVE" AS v
            JOIN LATERAL regexp_split_to_table(
                COALESCE(v."cwes", ''), '[[:space:]]*,[[:space:]]*'
            ) AS mapping(cwe_id) ON TRUE
            JOIN public."CWE" AS w
              ON mapping.cwe_id = 'CWE-' || w."CWE-ID"::text
            WHERE w."CWE-ID" = {cwe_number}
            ORDER BY v."dateAdded"::date DESC
            LIMIT 20
        '''
    if "랜섬웨어" in q:
        return '''
            SELECT "cveID", "vendorProject", "product", "vulnerabilityName", "dateAdded"
            FROM public."CVE"
            WHERE "knownRansomwareCampaignUse" = 'Known'
            ORDER BY "dateAdded"::date DESC
            LIMIT 20
        '''
    if ("CWE" in q.upper() or "취약점 유형" in q) and any(word in q for word in ("많", "빈도", "상위")):
        return '''
            SELECT 'CWE-' || w."CWE-ID"::text AS cwe_id,
                   w."Name" AS weakness_name,
                   COUNT(DISTINCT v."cveID") AS vulnerability_count
            FROM public."CVE" AS v
            JOIN LATERAL regexp_split_to_table(
                COALESCE(v."cwes", ''), '[[:space:]]*,[[:space:]]*'
            ) AS mapping(cwe_id) ON TRUE
            JOIN public."CWE" AS w
              ON mapping.cwe_id = 'CWE-' || w."CWE-ID"::text
            GROUP BY w."CWE-ID", w."Name"
            ORDER BY vulnerability_count DESC
            LIMIT 10
        '''
    year_match = re.search(r"(?<!\d)(20\d{2})(?!\d)", q)
    if year_match and any(word in q for word in ("월", "추이", "집계", "등록")):
        year = int(year_match.group(1))
        return f'''
            SELECT TO_CHAR("dateAdded"::date, 'YYYY-MM') AS month,
                   COUNT(*) AS vulnerability_count
            FROM public."CVE"
            WHERE EXTRACT(YEAR FROM "dateAdded"::date) = {year}
            GROUP BY month
            ORDER BY month
        '''
    if any(word in q for word in ("제조사", "벤더", "회사")) and any(word in q for word in ("많", "상위", "순위")):
        return '''
            SELECT "vendorProject", COUNT(*) AS vulnerability_count
            FROM public."CVE"
            GROUP BY "vendorProject"
            ORDER BY vulnerability_count DESC, "vendorProject"
            LIMIT 10
        '''
    return '''
        SELECT "cveID", "vendorProject", "product", "vulnerabilityName", "dateAdded"
        FROM public."CVE"
        ORDER BY "dateAdded"::date DESC
        LIMIT 10
    '''


def text_to_sql(question: str) -> str:
    question = question.strip()
    if not question:
        raise ValueError("질문은 비어 있을 수 없습니다.")

    if llm is not None:
        system_prompt = f'''
당신은 보안 취약점 데이터용 PostgreSQL 전문가입니다.
사용자의 한국어 질문을 아래 실제 스키마에 맞는 SQL 하나로 변환하세요.

[실제 스키마]
{SCHEMA_DESCRIPTION}

[관계 규칙]
{JOIN_GUIDE}

데이터 의미:
- CVE 테이블은 CISA Known Exploited Vulnerabilities(KEV) 목록이다.
- 따라서 CVE의 모든 행은 이미 실제 악용이 확인된 취약점이다.
- 질문에 '실제 악용'이 있어도 vulnerabilityName이나 shortDescription에서 해당 문구를 검색하지 않는다.
- '실제로 악용된 취약점이 가장 많은 제조사'는 CVE 전체를 vendorProject로 GROUP BY 한다.
- knownRansomwareCampaignUse는 실제 악용 여부가 아니라 랜섬웨어 캠페인 사용 여부다.

규칙:
- SELECT 또는 WITH 읽기 쿼리만 생성한다.
- 테이블과 컬럼은 표시된 대소문자를 유지하고 큰따옴표로 감싼다.
- 날짜 컬럼 dateAdded와 dueDate는 TEXT이므로 날짜 비교 시 ::date로 변환한다.
- CVE와 CWE를 JOIN할 때 제공된 LATERAL 분해 규칙을 그대로 사용한다.
- CVSS와 EPSS 컬럼은 없으므로 존재한다고 가정하지 않는다.
- 상세 목록은 LIMIT 100 이하로 제한한다.
- 설명이나 마크다운 없이 SQL만 반환한다.
'''
        try:
            response = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=question),
            ])
            return validate_read_only_sql(response.content) + ";"
        except Exception as error:
            print(f"⚠ LLM SQL 생성 실패, 규칙 기반 폴백 사용: {type(error).__name__}")

    return validate_read_only_sql(fallback_sql(question)) + ";"

print("✓ 동적 Text2SQL 함수 준비 완료")

## 6. SQL 실행과 자연어 답변 생성

In [ ]:
def dataframe_to_markdown(frame: pd.DataFrame, max_rows: int = 20) -> str:
    if frame.empty:
        return "조회 결과가 없습니다."
    shown = frame.head(max_rows).copy()
    shown = shown.map(
        lambda value: str(value).replace("|", "\\|").replace("\n", " ")[:160]
    )
    header = "| " + " | ".join(map(str, shown.columns)) + " |"
    separator = "| " + " | ".join(["---"] * len(shown.columns)) + " |"
    rows = ["| " + " | ".join(row) + " |" for row in shown.astype(str).values.tolist()]
    return "\n".join([header, separator, *rows])


def answer_from_result(question: str, sql: str, result: pd.DataFrame) -> str:
    table_text = dataframe_to_markdown(result)
    if llm is not None:
        try:
            response = llm.invoke([
                SystemMessage(content=(
                    "당신은 보안 취약점 데이터 분석가입니다. SQL 결과에 있는 사실만 사용해 "
                    "한국어로 핵심 결론을 먼저 설명하고, 수치와 CVE/CWE ID를 정확히 보존하세요."
                )),
                HumanMessage(content=f"질문: {question}\n\nSQL: {sql}\n\n결과:\n{table_text}"),
            ])
            return response.content
        except Exception as error:
            print(f"⚠ LLM 답변 생성 실패, 표 형식 답변 사용: {type(error).__name__}")
    return f"조회 결과는 {len(result)}행입니다.\n\n{table_text}"


def query_database(question: str) -> dict:
    """LLM SQL을 우선 사용하고, 실행 실패나 빈 결과면 검증 SQL로 한 번 재시도한다."""
    sql = text_to_sql(question)
    used_fallback = False
    execution_error = None
    try:
        result = run_read_only_sql(sql)
    except Exception as error:
        execution_error = error
        result = pd.DataFrame()

    verified_sql = validate_read_only_sql(fallback_sql(question)) + ";"
    should_retry = (execution_error is not None or result.empty) and (
        clean_sql(sql) != clean_sql(verified_sql)
    )
    if should_retry:
        reason = type(execution_error).__name__ if execution_error else "빈 결과"
        print(f"⚠ 생성 SQL 결과가 유효하지 않아 검증 SQL로 재시도합니다: {reason}")
        sql = verified_sql
        result = run_read_only_sql(sql)
        used_fallback = True
    elif execution_error is not None:
        raise execution_error

    answer = answer_from_result(question, sql, result)
    return {
        "question": question,
        "sql": sql,
        "result": result,
        "answer": answer,
        "used_fallback": used_fallback,
    }

print("✓ 자연어 질문 → 안전한 SQL → 실행 → 답변 파이프라인 준비 완료")

## 7. 완전한 Text2SQL 1건 테스트

In [ ]:
question = "실제로 악용된 취약점이 가장 많은 제조사 10곳을 알려줘"
test_run = query_database(question)
print(f"질문: {test_run['question']}\n")
print("생성 SQL:")
print(test_run["sql"])
print("\nSQL 결과:")
display(test_run["result"])
print("\n답변:")
display(Markdown(test_run["answer"]))
assert not test_run["result"].empty
print("\n✓ 완전한 Text2SQL 테스트 통과")

## 8. 다양한 자연어 질문 테스트

In [ ]:
questions = [
    "실제로 악용된 취약점이 가장 많은 제조사 10곳을 알려줘",
    "랜섬웨어 캠페인에 사용된 것으로 확인된 최근 취약점을 알려줘",
    "실제 악용 취약점과 가장 많이 연결된 CWE 유형 10개는?",
    "2026년에 등록된 취약점을 월별로 집계해줘",
    "CWE-89에 해당하는 실제 악용 취약점을 최근 순으로 보여줘",
]

text2sql_test_results = []
for index, question in enumerate(questions, start=1):
    print(f"\n{'=' * 100}\n[{index}] {question}\n{'=' * 100}")
    run = query_database(question)
    text2sql_test_results.append(run)
    print("SQL:")
    print(run["sql"])
    print(f"결과: {len(run['result'])}행")
    display(run["result"].head(10))
    display(Markdown(run["answer"]))

assert len(text2sql_test_results) == 5
assert all(validate_read_only_sql(run["sql"]) for run in text2sql_test_results)
assert all(not run["result"].empty for run in text2sql_test_results)
print("\n✓ 자연어 질문 5개 생성·검증·실행 테스트 통과")